# 09e: LightGBM and random forest on Cardio-70k and UCI Heart (E1)

Design decisions
- Completes the main results of the plan (E1: four tasks, three model families). CDC Diabetes and BRFSS Heart have all three families (notebooks 09 and 09b); this notebook adds LightGBM and random forest to Cardio-70k and to UCI Heart.
- Cardio-70k follows notebook 09b on the frozen Cardio design of notebook 09c: the same training portion, the 2,000 prior patients and lambda, M, feature-order seed and search budget, the explained subsamples of calibration split 2 and of the test split, calibration split 1 for certificates, the targets (alpha_0 = 0.20, beta = 0.80, the scaled grid and the plan's grid) and alpha_sel. Only the model changes: LightGBM and random forest with the settings closest to the frozen XGBoost, each fitted on the training portion with a Youden threshold on 5-fold out-of-fold scores, exported, checked against the native library on every training row, saved and hashed before calibration split 1 is read. Every box that could be released at the loosest target is re-checked by the second exact check; release, the policy test, test results and the consistency check follow notebook 09d.
- UCI Heart follows the leave-one-site-out design of notebook 13b on the same rows: for each held-out site, the training, calibration and test rows of the other three sites from the same split seed; the prior from the training rows; settings chosen on the training rows by 5-fold out-of-fold AUROC over the grid of the XGBoost models (depth 2, 3 or 4; at least 1 or 5 records per leaf; LightGBM with 100 trees, 32 bins and learning rate 0.3, random forest with 50 trees); the frozen heart targets (alpha_0 = 0.05, alpha_1 = 0.70) and the lenient grid point (alpha_0 = 0.10, alpha_1 = 0.80); every record of the held-out site is explained and checked; no policy test (no second calibration split at this size).
- The XGBoost rows are those of notebooks 09d (Cardio-70k) and 13b (UCI). Each unit is saved on Drive when finished, and candidate sets every 500 new cells, so an interrupted run resumes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time, zlib
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits, uci
from scax.experiments.main_results import FrozenTask
from scax.experiments import model_ablations as ma
from scax.experiments import shift as sh
from scax.experiments import families as fam
from scax.metrics.model_metrics import model_metrics
from scax.models.export import Ensemble

SEED = 42
seed_everything(SEED)
FAMILIES = ['lightgbm', 'sklearn_rf']
LIMIT = None          # every frozen explained patient
LOG = lambda m: print(m, flush=True)
OUT_DIR = paths.DERIVED / 'families'; OUT_DIR.mkdir(parents=True, exist_ok=True)
UCI_DIR = paths.DERIVED / 'families_uci'; UCI_DIR.mkdir(parents=True, exist_ok=True)
REC09C, REC09D = json.load(open(f'{paths.RESULTS}/09c_frozen.json')), json.load(open(f'{paths.RESULTS}/09d_results.json'))
assert REC09C['written_before_calibration_split_1']
assert LIMIT is not None or REC09D['limit'] is None, 'notebook 09d must have run on every explained patient'
CT = 'cardio70k'
FTC = FrozenTask(CT, REPO_DIR, DRIVE_ROOT, REC09C['frozen_config_sha256'][CT])
REC05, REC03 = json.load(open(f'{paths.RESULTS}/05_frozen.json')), json.load(open(f'{paths.RESULTS}/03_nhanes_uci.json'))
FTH = FrozenTask('brfss2015_heart', REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256']['brfss2015_heart'])
UCI_FILE = paths.DERIVED / 'uci' / 'uci_heart_sites.parquet'
assert sha256_file(UCI_FILE) == REC03['uci']['parquet_sha256'], 'the UCI table differs from the one notebook 03 wrote'
X, y = public.LOADERS[CT](paths.RAW)
XC, yC, NAMES_C = X.to_numpy(dtype=float), y, list(X.columns)
del X
code = np.load(paths.DERIVED / 'splits' / f'{CT}.npy')
assert splits.summary(code)['sha256'] == FTC.cfg['split_sha256'], 'split differs from the frozen one'
PRIOR_ROWS = np.array(json.load(open(DRIVE_ROOT + '/' + FTC.cfg['prior']['file']))['training_rows'])
assert len(PRIOR_ROWS) == FTC.cfg['prior']['training_patients']
print('Cardio-70k frozen configuration', FTC.cfg_sha[:16], '| targets', {k: FTC.targets[k] for k in ('alpha_0', 'beta', 'alpha_sel')},
      '| explained', len(FTC.cal2_rows), len(FTC.test_rows))

## Cardio-70k: LightGBM and random forest on the frozen design

In [ ]:
def lim(a):
    return a if LIMIT is None else a[:LIMIT]


def cardio_unit(family):
    """One model family on the frozen Cardio design; saved on Drive when finished."""
    res_file = OUT_DIR / f'{CT}__{family}__results.json'
    if res_file.exists():
        print(f'{CT} | {family}: loaded from Drive', flush=True)
        return json.load(open(res_file))
    t0 = time.time()
    T = FTC.targets
    tr = splits.indices(code, 'train')
    params = fam.family_params(FTC.cfg['model']['params'], NAMES_C, family)
    model, ens, pred = ma.fit_variant(family, XC[tr], yC[tr], params, SEED, NAMES_C)
    model_file = OUT_DIR / f'{CT}__{family}__model.json'
    if model_file.exists():
        assert Ensemble.load(model_file).hash() == ens.hash(), f'{CT} {family}: refitting does not reproduce the saved model'
    else:
        ens.save(model_file)
    agree = ma.exported_vs_native_detail(model, family, pred, XC[tr])
    assert agree['beyond_rounding'] == 0, f'{CT} {family}: exported and native models disagree: {agree}'
    c = FTC.cfg['candidates']
    sm = sh.ShiftModel(f'{CT} {family}', pred, c['M'], c['order_seed'], c['node_budget'], FTC.lam,
                       {0: T['alpha_0'], 1: T['alpha_1']}, T['delta_1'], cache_path=OUT_DIR / f'{CT}__{family}__candidates.pkl')
    sm.set_prior(XC[lim(PRIOR_ROWS)], log=LOG)
    write_json(OUT_DIR / f'{CT}__{family}__before_calibration.json', {
        'model_sha256': ens.hash(), 'tau': float(ens.tau), 'settings': params, 'exported_vs_native_training': agree,
        'prior_boxes': len(sm.counts), 'lambda': FTC.lam})          # fixed before calibration split 1 is read
    cal1 = splits.indices(code, 'cal1')
    sm.set_calibration(XC[cal1], yC[cal1])
    c2, te = lim(FTC.cal2_rows), lim(FTC.test_rows)
    # eligibility and the second check at the loosest target of either grid (0.30 and 0.80, as in notebook 09d)
    loosest = {**T, 'alpha_0_grid': sorted(set(T['alpha_0_grid'] + T['plan_alpha_0_grid'])),
               'beta_grid': sorted(set(T['beta_grid'] + T['plan_beta_grid']))}
    run = fam.FamilyRun(sm, loosest, XC[c2], yC[c2], XC[te], yC[te], log=LOG)
    rows, main = [], None
    for grid, a0s, betas in (('scaled', T['alpha_0_grid'], T['beta_grid']), ('plan', T['plan_alpha_0_grid'], T['plan_beta_grid'])):
        run.T = {**T, 'alpha_0_grid': a0s, 'beta_grid': betas}
        g_rows, g_main = run.grid_rows()
        rows += [{**r, 'grid': grid, 'frozen_targets': bool(r['frozen_targets'] and grid == 'scaled')} for r in g_rows]
        if grid == 'scaled':
            main = g_main
    run.T = T
    test_all = splits.indices(code, 'test')
    cons = run.consistency(main, sm.g.cells(XC[test_all]), yC[test_all])
    mm = model_metrics(model.predict_proba(XC[test_all])[:, 1], yC[test_all], pred.predict(XC[test_all]))
    r = {'task': CT, 'family': family, 'settings': params, 'model_sha256': ens.hash(), 'tau': float(ens.tau),
         'exported_vs_native_training': agree, 'thresholds': int(sum(len(t) for t in sm.g.thresholds)),
         'prior_boxes': len(sm.counts), 'boxes': len(run.keys), 'eligible_boxes': run.eligible,
         'unverified_boxes': run.unverified, 'rows': rows, 'consistency': cons, 'model_metrics': mm,
         'median_seconds_per_cell': float(np.median(list(sm.gen.seconds.values()))), 'seconds': time.time() - t0}
    write_json(res_file, r)
    fr = [x for x in rows if x['frozen_targets']]
    print(f"{CT} | {family}: AUROC {mm['auroc']:.3f}; " + '; '.join(
        f"{x['kind']}: {x['test_certified_share']:.1%} of test patients, error {x['test_error_rate']:.1%}, "
        f"policy {'certified' if x['policy_certified'] else 'not certified'} (U_sel {x['U_sel']:.3f} against {x['alpha_sel']})"
        for x in fr) + f"; {time.time() - t0:.0f}s", flush=True)
    return r


CARDIO = {f: cardio_unit(f) for f in FAMILIES}

## UCI Heart: each site held out, LightGBM and random forest

In [ ]:
U = pd.read_parquet(UCI_FILE)
XU, yU, SITE = U[uci.PREDICTORS].to_numpy(float), U[uci.TARGET].to_numpy(int), U.site.to_numpy()
CAND = FTH.cfg['candidates']
ALPHA = {0: FTH.targets['alpha_0'], 1: FTH.targets['alpha_1']}
LENIENT = {0: 0.10, 1: 0.80}
SETS = (('frozen', ALPHA), ('lenient', LENIENT))
GRID = {'lightgbm': [{'n_estimators': 100, 'max_bin': 32, 'learning_rate': 0.3, 'max_depth': d, 'num_leaves': 2 ** d - 1,
                      'min_child_samples': w} for d in (2, 3, 4) for w in (1, 5)],
        'sklearn_rf': [{'n_estimators': 50, 'max_depth': d, 'min_samples_leaf': w} for d in (2, 3, 4) for w in (1, 5)]}


def seed_of(*parts):
    return zlib.crc32('|'.join(map(str, parts)).encode()) % (2 ** 31)


def uci_unit(held, family):
    """One held-out site and model family; the rows of the other sites split as in notebook 13b."""
    res_file = UCI_DIR / f'uci_without_{held}__{family}__results.json'
    if res_file.exists():
        print(f'UCI without {held} | {family}: loaded from Drive', flush=True)
        return json.load(open(res_file))
    t0 = time.time()
    name = f'uci_without_{held}'
    idx = np.flatnonzero(SITE != held)
    part = sh.split_small(yU[idx], seed_of('uci split', name))          # the split of notebook 13b
    tr_, cal_, te_ = (idx[part == k] for k in (0, 1, 2))
    best, table = sh.choose_settings(XU[tr_], yU[tr_], GRID[family], SEED, family=family)
    model, ens, pred = ma.fit_variant(family, XU[tr_], yU[tr_], best, SEED, uci.PREDICTORS)
    f = UCI_DIR / f'{name}__{family}__model.json'
    if f.exists():
        assert Ensemble.load(f).hash() == ens.hash(), f'{name} {family}: refitting does not reproduce the saved model'
    else:
        ens.save(f)
    agree = ma.exported_vs_native_detail(model, family, pred, XU[np.r_[tr_, cal_]])
    assert agree['beyond_rounding'] == 0, f'{name} {family}: exported and native models disagree: {agree}'
    sm = sh.ShiftModel(f'{name} {family}', pred, CAND['M'], CAND['order_seed'], CAND['node_budget'], FTH.lam, ALPHA,
                       FTH.targets['delta_1'], cache_path=UCI_DIR / f'{name}__{family}__candidates.pkl')
    sm.set_prior(XU[tr_], log=LOG)
    sm.set_calibration(XU[cal_], yU[cal_])
    rows = []
    for target, role, m in ((f'{uci.SITE_NAMES[held]} held out: source test split', 'reference', te_),
                            (f'{uci.SITE_NAMES[held]} held out', 'target', np.flatnonzero(SITE == held))):
        mm = model_metrics(model.predict_proba(XU[m])[:, 1], yU[m], pred.predict(XU[m]))
        for set_name, alpha in SETS:
            te = sh.TargetEval(sm, XU[m], yU[m], XU[m], yU[m], log=None, alpha=alpha)
            met = te.transport()
            for k in (0, 1):
                rows.append({'setting': 'uci_main', 'model': name, 'family': family, 'target': target, 'role': role,
                             'targets_set': set_name, 'alpha_claim': alpha[k], 'kind': k, 'explained': int(len(m)),
                             'auroc': mm['auroc'], 'prevalence': mm['prevalence'], **met[k]})
    r = {'model': name, 'held_out': held, 'family': family, 'settings': best, 'settings_table': table.to_dict('records'),
         'model_sha256': ens.hash(), 'tau': float(ens.tau), 'exported_vs_native': agree,
         'training_records': int(len(tr_)), 'calibration_records': int(len(cal_)), 'prior_boxes': len(sm.counts),
         'rows': rows, 'seconds': time.time() - t0}
    write_json(res_file, r)
    tgt = [x for x in rows if x['role'] == 'target' and x['kind'] == 1]
    print(f"UCI without {held} | {family}: depth {best['max_depth']}, AUROC on the held-out site {tgt[0]['auroc']:.3f}; enrichment claims "
          + ', '.join(f"{x['targets_set']}: {x['coverage']:.1%} of patients, {x['error_rate'] if x['accepted'] else float('nan'):.1%} without disease"
                      for x in tgt) + f'; {time.time() - t0:.0f}s', flush=True)
    return r


UCIR = {(h, f): uci_unit(h, f) for h in uci.SITES for f in FAMILIES}

## Results, with the XGBoost rows of notebooks 09d and 13b

In [ ]:
pd.set_option('display.width', 250)
MAIN_C = pd.DataFrame([{'family': f, **row} for f, r in CARDIO.items() for row in r['rows']])
X09D = pd.read_csv(f'{paths.RESULTS}/09d_main.csv').assign(family='xgboost')
ALL_C = pd.concat([X09D, MAIN_C.assign(task=CT)], ignore_index=True)
ALL_C['family_name'] = ALL_C.family.map(fam.FAMILY_NAMES)
MET_C = pd.concat([pd.read_csv(f'{paths.RESULTS}/09d_model_metrics.csv').assign(family='xgboost'),
                   pd.DataFrame([{'task': CT, 'family': f, **r['model_metrics']} for f, r in CARDIO.items()])], ignore_index=True)
CONS_C = pd.DataFrame([{'task': CT, 'family': f, **c} for f, r in CARDIO.items() for c in r['consistency']])
COLS = ['family_name', 'kind', 'test_certified_share', 'test_certified_share_of_predicted_class', 'test_error_rate',
        'test_error_upper_95', 'mean_size', 'median_calibration_support', 'cal2_accepted', 'U_sel', 'alpha_sel', 'policy_certified']
print('Cardio-70k, frozen targets (alpha_0 = 0.20, beta = 0.80):')
print(ALL_C[ALL_C.frozen_targets == True][COLS].sort_values(['kind', 'family_name']).round(4).to_string(index=False))
print('\nCardio-70k, underlying models on the full test split:')
print(MET_C[['family', 'auroc', 'auprc', 'brier', 'ece_15_equal_mass', 'calibration_slope', 'sensitivity', 'specificity']].round(4).to_string(index=False))
if len(CONS_C):
    print('\nCardio-70k consistency check (flagged = corrected lower limit above the target):')
    print(CONS_C.groupby(['family', 'box_class']).agg(released_boxes=('test_n', 'size'), mean_test_error=('test_error', 'mean'),
                                                      flagged=('estimate_above_target', 'sum')).to_string())
print('\nCardio-70k boxes, second check and runtime:')
print(pd.DataFrame([{'family': f, 'thresholds': r['thresholds'], 'boxes': r['boxes'], 'eligible': r['eligible_boxes'],
                     'unverified': r['unverified_boxes'], 'median_seconds_per_cell': r['median_seconds_per_cell'],
                     'hours': r['seconds'] / 3600} for f, r in CARDIO.items()]).round(3).to_string(index=False))

UCI_X = pd.read_csv(f'{paths.RESULTS}/13b_shift.csv')
UCI_X = UCI_X[UCI_X.setting == 'uci_main'].assign(family='xgboost')
UCI_T = pd.read_csv(f'{paths.RESULTS}/13b_shift_targets.csv')
UCI_X = UCI_X.merge(UCI_T[['model', 'target', 'targets_set', 'auroc', 'prevalence']], on=['model', 'target', 'targets_set'])
UCI_F = pd.DataFrame([row for r in UCIR.values() for row in r['rows']])
UCI_ALL = pd.concat([UCI_X, UCI_F], ignore_index=True)
UCI_ALL['family_name'] = UCI_ALL.family.map(fam.FAMILY_NAMES)
UCI_ALL['kind_name'] = UCI_ALL.kind.map(fam.KIND_NAMES)
UC = ['target', 'family_name', 'targets_set', 'kind_name', 'auroc', 'explained', 'accepted', 'coverage', 'errors', 'error_rate',
      'error_upper_95', 'refuted_share', 'certificate_refuted_share', 'mean_size']
print('\nUCI Heart, each site held out (targets: frozen alpha_0 0.05 / alpha_1 0.70; lenient 0.10 / 0.80):')
print(UCI_ALL[UCI_ALL.role == 'target'][UC].sort_values(['target', 'targets_set', 'kind_name', 'family_name']).round(3).to_string(index=False))
print('\nUCI settings chosen on training rows (out-of-fold AUROC):')
print(pd.DataFrame([{'model': r['model'], 'family': r['family'], **r['settings'], 'oof_auroc': max(t['oof_auroc'] for t in r['settings_table'])}
                    for r in UCIR.values()]).round(3).to_string(index=False))

## Figure: certified share by model family on Cardio-70k and UCI Heart

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7.5, 'axes.spines.top': False, 'axes.spines.right': False})
STYLE = {'xgboost': ('#0072B2', 'o'), 'lightgbm': ('#D55E00', 's'), 'sklearn_rf': ('#009E73', '^')}
T = FTC.targets
fig, axes = plt.subplots(2, 2, figsize=(7.4, 5.6))
for fmy, (col, mk) in STYLE.items():
    d = ALL_C[ALL_C.family == fmy]
    neg = d[d.kind == 'negative outcome'].drop_duplicates('alpha_0').sort_values('alpha_0')   # depends on alpha_0 only
    enr = d[d.kind == 'enrichment'].drop_duplicates('beta').sort_values('beta')               # depends on beta only
    axes[0, 0].plot(neg.alpha_0, neg.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.3, label=fam.FAMILY_NAMES[fmy])
    axes[0, 1].plot(enr.beta, enr.test_certified_share, color=col, marker=mk, ms=3.5, lw=1.3)
for ax, x0 in ((axes[0, 0], T['alpha_0']), (axes[0, 1], T['beta'])):
    ax.axvline(x0, color='0.3', ls='--', lw=0.8)
    ax.set_ylim(bottom=0); ax.grid(color='0.92', lw=0.6); ax.set_axisbelow(True)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
axes[0, 0].set_xlabel(r'Target error for negative-outcome claims ($\alpha_0$)')
axes[0, 1].set_xlabel(r'Claimed minimum share with the outcome ($\beta$)')
axes[0, 0].set_ylabel('Cardio-70k: share of test patients')
axes[0, 0].set_title('Certified negative-outcome explanations', fontsize=8.5)
axes[0, 1].set_title('Certified enrichment explanations', fontsize=8.5)
sites = [uci.SITE_NAMES[s] for s in uci.SITES]
w = 0.26
for j, (fmy, (col, mk)) in enumerate(STYLE.items()):
    d = UCI_ALL[(UCI_ALL.role == 'target') & (UCI_ALL.kind == 1) & (UCI_ALL.family == fmy) & (UCI_ALL.targets_set == 'frozen')]
    d = d.set_index(d.target.str.replace(' held out', '', regex=False)).reindex(sites)
    xs = np.arange(len(sites)) + (j - 1) * w
    axes[1, 0].bar(xs, d.coverage, width=w, color=col, edgecolor='white', linewidth=0.5)
    ok = d.accepted.fillna(0).to_numpy() > 0
    prec, lo = 1 - d.error_rate.to_numpy(), 1 - d.error_upper_95.to_numpy()
    axes[1, 1].errorbar(xs[ok], prec[ok], yerr=[prec[ok] - lo[ok], np.zeros(ok.sum())], fmt=mk, ms=4, color=col, elinewidth=0.8,
                        capsize=2.5, capthick=0.8)
axes[1, 1].axhline(1 - ALPHA[1], color='0.3', ls='--', lw=0.8)
for ax in axes[1]:
    ax.set_xticks(np.arange(len(sites))); ax.set_xticklabels([s.replace('VA Long Beach', 'VA Long\nBeach') for s in sites])
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0)); ax.grid(axis='y', color='0.92', lw=0.6); ax.set_axisbelow(True)
    ax.set_xlabel('UCI Heart: site held out')
axes[1, 0].set_ylabel('Share of held-out patients with a\ncertified enrichment explanation')
axes[1, 1].set_ylabel('Share with angiographic disease among them\n(one-sided 95% lower bound)')
axes[1, 1].set_ylim(0, 1.02)
axes[1, 0].set_title(r'UCI Heart, enrichment claims ($\beta$ = 30%)', fontsize=8.5)
axes[1, 1].set_title('UCI Heart, claim checked on the held-out site', fontsize=8.5)
h, l = axes[0, 0].get_legend_handles_labels()
from matplotlib.lines import Line2D
h.append(Line2D([], [], color='0.3', ls='--', lw=0.8)); l.append('Frozen target / claimed minimum share')
fig.legend(h, l, loc='upper center', ncol=4, frameon=False, fontsize=7.5, bbox_to_anchor=(0.5, 1.01))
fig.tight_layout(rect=(0, 0, 1, 0.95))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_families_cardio_uci.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
ALL_C.to_csv(f'{paths.RESULTS}/09e_cardio_main.csv', index=False)
MET_C.to_csv(f'{paths.RESULTS}/09e_cardio_model_metrics.csv', index=False)
CONS_C.to_csv(f'{paths.RESULTS}/09e_cardio_box_consistency.csv', index=False)
UCI_ALL.to_csv(f'{paths.RESULTS}/09e_uci_main.csv', index=False)
write_json(f'{paths.RESULTS}/09e_families.json', {
    'seed': SEED, 'limit': LIMIT, 'cardio_frozen_config_sha256': FTC.cfg_sha, 'heart_frozen_config_sha256': FTH.cfg_sha,
    'uci_parquet_sha256': REC03['uci']['parquet_sha256'], 'uci_grid': GRID,
    'cardio': {f: {k: v for k, v in r.items() if k not in ('rows', 'consistency')} for f, r in CARDIO.items()},
    'cardio_policy': {f: {x['kind']: {'certified': x['policy_certified'], 'U_sel': x['U_sel'], 'alpha_sel': x['alpha_sel'],
                                      'accepted': x['cal2_accepted']} for x in r['rows'] if x['frozen_targets']} for f, r in CARDIO.items()},
    'uci': {f'{h}__{f}': {k: v for k, v in r.items() if k != 'rows'} for (h, f), r in UCIR.items()}})
print('written: results/09e_cardio_main.csv, 09e_cardio_model_metrics.csv, 09e_cardio_box_consistency.csv, 09e_uci_main.csv, '
      '09e_families.json; figures/fig_families_cardio_uci')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "09e: LightGBM and random forest on Cardio-70k (frozen design, policy test) and UCI Heart (each site held out), with the XGBoost rows of 09d and 13b, figure; completes E1"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)